# H-007 Baseline Experiment


Evaluation note: no separate IS/OOS split convention exists in repo truth. The baseline uses the full registered evaluation range for both `in_sample_sharpe` and `oos_sharpe`, matching the current runbook baseline pattern.

Gate 2 semantics note: this notebook reports Gate 2 according to current `lib/evaluation.py` repo truth. The displayed legacy `GateReport` wording may read `DSR > 1.0 at 95% CI`, but the evaluator currently checks whether `dsr_val` exceeds 0.95 against its expected maximum-SR benchmark from the supplied trial count.

Trial log note: WP-4b1 appended the trial row with `make_trial_entry()` and `append_trial()` during package execution. Re-running this notebook recomputes evidence but does not append another trial row.


In [1]:
from pathlib import Path
import json
import hashlib
import math
import pandas as pd
from lib.backtest import run_backtest
from lib.evaluation import check_promotion_gates, cost_drag, hit_rate, max_drawdown, profit_factor, sharpe

HYPOTHESIS_ID = "H-007"
TRIAL_ID = "b88b2e9a-03b3-41de-8197-dee370a9c536"
TRIAL_COUNT_BEFORE_APPEND = 6
DATASET_IDS = ['binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z', 'binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z']
PARAMS = {'funding_lookback': 6, 'funding_threshold': 0.0001, 'momentum_lookback': 12, 'momentum_threshold': 0.0}
COSTS = {'fee_bps_taker': 4.0, 'fee_bps_maker': 2.0, 'slippage_bps': 2.0, 'initial_equity': 10000.0}
OHLCV_4H_PATH = Path("data/processed/binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet")
FUNDING_PATH = Path("data/processed/binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z.parquet")

def _sha256_of_dict(d):
    return hashlib.sha256(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()

def load_ohlcv(path):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df["close_time_ms"].astype("int64"), unit="ms", utc=True)
    return df[["open", "high", "low", "close", "volume"]].astype(float).sort_index()

def load_kline(path, name):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df["close_time_ms"].astype("int64"), unit="ms", utc=True)
    out = df[["open", "high", "low", "close", "volume"]].astype(float).sort_index()
    out.columns = [f"{name}_{col}" for col in out.columns]
    return out

def load_funding():
    df = pd.read_parquet(FUNDING_PATH).copy()
    df.index = pd.to_datetime(df["fundingTime"].astype("int64"), unit="ms", utc=True)
    return df.rename(columns={"fundingRate": "funding_rate"})[["funding_rate"]].astype(float).sort_index()

def filter_applicable_funding(signal, ohlcv, funding):
    common_idx = signal.index.intersection(ohlcv.index)
    sig = signal.reindex(common_idx).fillna(0).astype(int)
    bars = list(common_idx)
    position = 0
    event_ts = []
    funding_idx = funding.index
    for i, bar_ts in enumerate(bars):
        if i > 0 and position != 0:
            prev_bar_ts = bars[i - 1]
            matches = funding_idx[(funding_idx > prev_bar_ts) & (funding_idx <= bar_ts)]
            event_ts.extend(matches)
        if i > 0:
            desired = int(sig.iloc[i - 1])
            if desired != position:
                position = desired
    if not event_ts:
        return funding.iloc[0:0].copy()
    return funding.loc[pd.DatetimeIndex(sorted(set(event_ts)))]

def run_candidate(signal, ohlcv, funding):
    applicable = filter_applicable_funding(signal, ohlcv, funding)
    result = run_backtest(signal, ohlcv, applicable, **COSTS, execution="next_bar_open", stream="klines", interval="4h")
    return result, len(applicable)

def summarize(result):
    returns = result.equity_curve.pct_change().dropna()
    mdd, peak, trough = max_drawdown(result.equity_curve)
    cd = cost_drag(result)
    return {"final_equity": float(result.equity_curve.iloc[-1]), "total_pnl": float(result.equity_curve.iloc[-1] - result.equity_curve.iloc[0]), "sharpe": float(sharpe(returns, "4h")), "max_drawdown": float(mdd), "max_drawdown_peak": str(peak) if peak is not None else None, "max_drawdown_trough": str(trough) if trough is not None else None, "n_obs": int(len(returns)), "n_trades": int(len(result.trades_df)), "hit_rate": float(hit_rate(result.trades_df)), "profit_factor": float(profit_factor(result.trades_df)), "fees_paid": float(result.fees_paid), "slippage_paid": float(result.slippage_paid), "funding_paid": float(result.funding_paid), "gross_pnl": float(cd["gross_pnl"]), "cost_drag_fraction": float(cd["cost_drag_fraction"]) if math.isfinite(cd["cost_drag_fraction"]) else None, "skipped_bars": int(result.skipped_bars)}

def side_summary(result):
    if result.trades_df.empty:
        return []
    return [{"direction": int(direction), "count": int(len(group)), "net_pnl_sum": float(group["net_pnl"].sum()), "net_pnl_mean": float(group["net_pnl"].mean())} for direction, group in result.trades_df.groupby("direction")]

def sparkline(series, width=72):
    vals = series.dropna()
    if vals.empty:
        return ""
    if len(vals) > width:
        vals = vals.iloc[::max(1, len(vals) // width)].iloc[:width]
    chars = "._-~=+*#%@"
    lo, hi = float(vals.min()), float(vals.max())
    if hi == lo:
        return chars[0] * len(vals)
    return "".join(chars[min(len(chars) - 1, int((float(v) - lo) / (hi - lo) * (len(chars) - 1)))] for v in vals)

def drawdown_series(equity):
    return (equity - equity.cummax()) / equity.cummax()

def h007_features(ohlcv, funding, funding_lookback=6, momentum_lookback=12):
    funding_aligned = funding["funding_rate"].reindex(ohlcv.index.union(funding.index)).sort_index().ffill().reindex(ohlcv.index)
    rolling_funding = funding_aligned.rolling(int(round(funding_lookback)), min_periods=int(round(funding_lookback))).mean()
    momentum = ohlcv["close"].pct_change(int(round(momentum_lookback)))
    return pd.DataFrame({"funding_aligned": funding_aligned, "rolling_funding": rolling_funding, "momentum": momentum}, index=ohlcv.index)

def h007_signal(ohlcv, funding, funding_lookback=6, funding_threshold=0.0001, momentum_lookback=12, momentum_threshold=0.0):
    features = h007_features(ohlcv, funding, funding_lookback, momentum_lookback)
    signal = pd.Series(0.0, index=ohlcv.index, name="signal")
    signal[(features["rolling_funding"] < -float(funding_threshold)) & (features["momentum"] > float(momentum_threshold))] = 1.0
    signal[(features["rolling_funding"] > float(funding_threshold)) & (features["momentum"] < -float(momentum_threshold))] = -1.0
    signal[features["rolling_funding"].isna() | features["momentum"].isna()] = 0.0
    return signal, features

def h010_features(ohlcv, mark, index, premium, basis_lookback=90):
    missing = {"mark_missing": int(len(ohlcv.index.difference(mark.index))), "index_missing": int(len(ohlcv.index.difference(index.index))), "premium_missing": int(len(ohlcv.index.difference(premium.index)))}
    assert not any(missing.values()), missing
    mark = mark.reindex(ohlcv.index)
    index = index.reindex(ohlcv.index)
    premium = premium.reindex(ohlcv.index)
    basis = (mark["mark_close"] - index["index_close"]) / index["index_close"]
    lb = int(round(basis_lookback))
    basis_z = (basis - basis.rolling(lb, min_periods=lb).mean()) / basis.rolling(lb, min_periods=lb).std()
    return pd.DataFrame({"basis": basis, "basis_z": basis_z, "premium_close": premium["premium_close"]}, index=ohlcv.index)

def h010_signal(ohlcv, mark, index, premium, basis_lookback=90, basis_z_threshold=2.0, exit_z_threshold=0.5, max_holding_bars=18):
    features = h010_features(ohlcv, mark, index, premium, basis_lookback)
    signal = pd.Series(0.0, index=ohlcv.index, name="signal")
    current = 0.0
    holding_bars = 0
    for ts in ohlcv.index:
        z = features.loc[ts, "basis_z"]
        premium_close = features.loc[ts, "premium_close"]
        if pd.isna(z) or pd.isna(premium_close):
            current = 0.0
            holding_bars = 0
            signal.loc[ts] = current
            continue
        short_entry = (z > float(basis_z_threshold)) and (premium_close > 0)
        long_entry = (z < -float(basis_z_threshold)) and (premium_close < 0)
        if current == -1.0:
            holding_bars += 1
            if long_entry:
                current = 1.0
                holding_bars = 0
            elif z <= float(exit_z_threshold) or holding_bars >= int(round(max_holding_bars)):
                current = 0.0
                holding_bars = 0
        elif current == 1.0:
            holding_bars += 1
            if short_entry:
                current = -1.0
                holding_bars = 0
            elif z >= -float(exit_z_threshold) or holding_bars >= int(round(max_holding_bars)):
                current = 0.0
                holding_bars = 0
        elif short_entry:
            current = -1.0
            holding_bars = 0
        elif long_entry:
            current = 1.0
            holding_bars = 0
        signal.loc[ts] = current
    return signal, features

def sensitivity_h007(ohlcv, funding, base):
    variants = [("funding_lookback_up_25pct", {"funding_lookback": 7.5, "funding_threshold": 0.0001, "momentum_lookback": 12, "momentum_threshold": 0.0}), ("funding_lookback_down_25pct", {"funding_lookback": 4.5, "funding_threshold": 0.0001, "momentum_lookback": 12, "momentum_threshold": 0.0}), ("funding_threshold_up_25pct", {"funding_lookback": 6, "funding_threshold": 0.000125, "momentum_lookback": 12, "momentum_threshold": 0.0}), ("funding_threshold_down_25pct", {"funding_lookback": 6, "funding_threshold": 0.000075, "momentum_lookback": 12, "momentum_threshold": 0.0}), ("momentum_lookback_up_25pct", {"funding_lookback": 6, "funding_threshold": 0.0001, "momentum_lookback": 15, "momentum_threshold": 0.0}), ("momentum_lookback_down_25pct", {"funding_lookback": 6, "funding_threshold": 0.0001, "momentum_lookback": 9, "momentum_threshold": 0.0})]
    rows = []
    for name, params in variants:
        sig, _ = h007_signal(ohlcv, funding, **params)
        res, _ = run_candidate(sig, ohlcv, funding)
        sr = summarize(res)["sharpe"]
        rows.append({"variant": name, "params": params, "sharpe": sr, "ratio_to_base": sr / base if base else 0.0})
    return rows

def sensitivity_h010(ohlcv, funding, mark, index, premium, base):
    variants = [("basis_lookback_up_25pct", {"basis_lookback": 112.5, "basis_z_threshold": 2.0, "exit_z_threshold": 0.5, "max_holding_bars": 18}), ("basis_lookback_down_25pct", {"basis_lookback": 67.5, "basis_z_threshold": 2.0, "exit_z_threshold": 0.5, "max_holding_bars": 18}), ("basis_z_threshold_up_25pct", {"basis_lookback": 90, "basis_z_threshold": 2.5, "exit_z_threshold": 0.5, "max_holding_bars": 18}), ("basis_z_threshold_down_25pct", {"basis_lookback": 90, "basis_z_threshold": 1.5, "exit_z_threshold": 0.5, "max_holding_bars": 18}), ("exit_z_threshold_up_25pct", {"basis_lookback": 90, "basis_z_threshold": 2.0, "exit_z_threshold": 0.625, "max_holding_bars": 18}), ("exit_z_threshold_down_25pct", {"basis_lookback": 90, "basis_z_threshold": 2.0, "exit_z_threshold": 0.375, "max_holding_bars": 18}), ("max_holding_bars_up_25pct", {"basis_lookback": 90, "basis_z_threshold": 2.0, "exit_z_threshold": 0.5, "max_holding_bars": 22.5}), ("max_holding_bars_down_25pct", {"basis_lookback": 90, "basis_z_threshold": 2.0, "exit_z_threshold": 0.5, "max_holding_bars": 13.5})]
    rows = []
    for name, params in variants:
        sig, _ = h010_signal(ohlcv, mark, index, premium, **params)
        res, _ = run_candidate(sig, ohlcv, funding)
        sr = summarize(res)["sharpe"]
        rows.append({"variant": name, "params": params, "sharpe": sr, "ratio_to_base": sr / base if base else 0.0})
    return rows

ohlcv = load_ohlcv(OHLCV_4H_PATH)
funding = load_funding()

signal, features = h007_signal(ohlcv, funding, **PARAMS)
result, funding_events_applied = run_candidate(signal, ohlcv, funding)
metrics = summarize(result)
sensitivity = sensitivity_h007(ohlcv, funding, metrics['sharpe'])
params_meta = {"n_free_params": 4, "sensitivity_min_ratio": min(r["ratio_to_base"] for r in sensitivity), "is_sharpe": metrics["sharpe"]}
gate_report = check_promotion_gates(result=result, trial_count=TRIAL_COUNT_BEFORE_APPEND, params_meta=params_meta, freq="4h")
config = {"hypothesis_id": HYPOTHESIS_ID, "primary_timeframe": "4h", "execution": "next_bar_open", "costs": COSTS, "params": PARAMS}
report = {
    "hypothesis_id": HYPOTHESIS_ID,
    "trial_id": TRIAL_ID,
    "trial_count_before_append": TRIAL_COUNT_BEFORE_APPEND,
    "dataset_ids": DATASET_IDS,
    "dataset_sha256": _sha256_of_dict({"ids": sorted(DATASET_IDS)}),
    "config_sha256": _sha256_of_dict(config),
    "params": PARAMS,
    "funding_events_applied": funding_events_applied,
    "signal_counts": {str(k): int(v) for k, v in signal.value_counts().sort_index().items()},
    "rolling_funding_valid": int(features["rolling_funding"].notna().sum()),
    "metrics": metrics,
    "sensitivity": sensitivity,
    "sensitivity_min_ratio": params_meta["sensitivity_min_ratio"],
    "side_summary": side_summary(result),
    "equity_sparkline": sparkline(result.equity_curve),
    "drawdown_sparkline": sparkline(drawdown_series(result.equity_curve)),
    "gate_report": gate_report.to_dict(),
    "verdict": "PASS" if gate_report.overall_pass else "NO-GO",
}
print(json.dumps(report, indent=2, sort_keys=True))


{
  "config_sha256": "d84c705055025588153cf6fa847bbcd858056a8712c2453af21873b0f927723d",
  "data": {
    "funding_8h": {
      "first": "2019-09-10 08:00:00+00:00",
      "last": "2026-04-12 16:00:00+00:00",
      "path": "data/processed/binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z.parquet",
      "rows": 7220
    },
    "ohlcv_4h": {
      "first": "2020-01-01 03:59:59.999000+00:00",
      "last": "2026-04-11 23:59:59.999000+00:00",
      "path": "data/processed/binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet",
      "rows": 13758
    }
  },
  "dataset_ids": [
    "binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z",
    "binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z"
  ],
  "dataset_sha256": "9c52bdd9b90a5c4c3f3857ee39de6941f0bd6de038007dc712616f5d6618fb87",
  "drawdown_sparkline": "@%%%%%%%%%%#*._=%*%####%%%%%%%%%%%%%%%%%%%%%%%##+========---------------",
  "equity_sparkline": "~~~=====